# Logistic Regression: Intrusion Detection

**Goal:** Use one feature—the number of failed logins per minute—to estimate whether a record is normal (`0`) or an attack (`1`).

Run the notebook from top to bottom. For each required experiment, change **one setting at a time** in its own cell, then rerun the notebook from the data-generation cell onward. Record the results in the experiment log at the end.

### 1. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### 2. Set the number of normal records

In [ ]:
NORMAL_COUNT = 60

### 3. Set the number of attack records
Required comparison: try `15`, then `60`.

In [ ]:
ATTACK_COUNT = 60

### 4. Set the learning rate
Required comparison: `0.05`, `0.5`, and `2.0`.

In [ ]:
LEARNING_RATE = 0.5

### 5. Set training iterations

In [ ]:
ITERATIONS = 1000

### 6. Set the decision threshold
Required comparison: `0.25`, `0.5`, and `0.75`.

In [ ]:
THRESHOLD = 0.5

### 7. Choose a new record to classify

In [ ]:
NEW_RECORD = 5.0

### 8. Generate sample data

In [ ]:
rng = np.random.default_rng(0)
normal = np.clip(rng.normal(1.5, 1.3, NORMAL_COUNT), 0, None)
attack = np.clip(rng.normal(7.0, 3.0, ATTACK_COUNT), 0, None)

x = np.r_[normal, attack]
x_raw = x.copy()
y = np.r_[np.zeros(NORMAL_COUNT), np.ones(ATTACK_COUNT)]

print(f"Normal records: {NORMAL_COUNT}")
print(f"Attack records: {ATTACK_COUNT}")
print(f"Total records: {len(y)}")

### 9. Scale the feature

In [ ]:
mean = x.mean()
sd = x.std()
x = (x - mean) / sd

print(f"Feature mean: {mean:.2f}")
print(f"Feature standard deviation: {sd:.2f}")

### 10. Define the sigmoid function

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

### 11. Train the logistic-regression model

In [ ]:
w = 0.0
b = 0.0
costs = []

for _ in range(ITERATIONS):
    p = sigmoid(w * x + b)
    error = p - y

    w -= LEARNING_RATE * np.mean(error * x)
    b -= LEARNING_RATE * np.mean(error)

    p = np.clip(sigmoid(w * x + b), 1e-12, 1 - 1e-12)
    cost = np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p)))
    costs.append(cost)

print("Training complete.")

### 12. Predict labels and calculate accuracy

In [ ]:
p = sigmoid(w * x + b)
prediction = (p >= THRESHOLD).astype(int)
accuracy = np.mean(prediction == y) * 100

print(f"Weight: {w:.2f}, bias: {b:.2f}")
print(f"Final cost: {costs[-1]:.3f}")
print(f"Threshold: {THRESHOLD:.2f}")
print(f"Predicted attacks: {prediction.sum()} of {len(y)} records")
print(f"Training accuracy: {accuracy:.1f}%")

### 13. Predict a new record

In [ ]:
new_scaled = (NEW_RECORD - mean) / sd
new_probability = sigmoid(w * new_scaled + b)
new_prediction = int(new_probability >= THRESHOLD)
label = "Attack" if new_prediction == 1 else "Normal"

print(
    f"New record ({NEW_RECORD:g} failed logins): "
    f"P(attack)={new_probability:.2f}, prediction={new_prediction} ({label})"
)

### 14. Plot the training cost

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(costs, color="purple")
plt.xlabel("Training step")
plt.ylabel("Average log-loss")
plt.title("Training cost: lower is better")
plt.tight_layout()
plt.show()

### 15. Plot records and predicted attack probabilities

In [ ]:
order = np.argsort(x_raw)

plt.figure(figsize=(8, 4))
plt.scatter(x_raw[y == 0], y[y == 0], color="blue", label="Normal (0)")
plt.scatter(x_raw[y == 1], y[y == 1], color="red", label="Attack (1)")
plt.plot(x_raw[order], p[order], color="green", label="Attack probability")
plt.axhline(THRESHOLD, color="gray", linestyle="--", label="Decision threshold")
plt.xlabel("Failed logins per minute")
plt.ylabel("Probability / true label")
plt.ylim(-0.08, 1.08)
plt.title("Records and model predictions")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Required experiments

Run the notebook once with the default settings. Then change **one parameter at a time**, rerun from **Generate sample data** onward, and write down the displayed results. Return other settings to their baseline values before testing the next parameter.

Suggested sequence:

1. **Attack count:** compare `ATTACK_COUNT = 15` with `ATTACK_COUNT = 60`.
2. **Learning rate:** compare `LEARNING_RATE = 0.05`, `0.5`, and `2.0`.
3. **Threshold:** compare `THRESHOLD = 0.25`, `0.5`, and `0.75`.
4. **New record:** change `NEW_RECORD` to another failed-login count and describe the prediction.

### Experiment log

Fill in the observed values after each run. Use the same baseline settings except for the one parameter being tested.

| Test | Setting changed | Final cost | Training accuracy | Predicted attacks | Notes |
|---|---|---:|---:|---:|---|
| Baseline | Defaults |  |  |  |  |
| Attack count | `ATTACK_COUNT = 15` |  |  |  |  |
| Attack count | `ATTACK_COUNT = 60` |  |  |  |  |
| Learning rate | `LEARNING_RATE = 0.05` |  |  |  |  |
| Learning rate | `LEARNING_RATE = 0.5` |  |  |  |  |
| Learning rate | `LEARNING_RATE = 2.0` |  |  |  |  |
| Threshold | `THRESHOLD = 0.25` |  |  |  |  |
| Threshold | `THRESHOLD = 0.5` |  |  |  |  |
| Threshold | `THRESHOLD = 0.75` |  |  |  |  |

## Questions to answer

1. What changed when the number of attack records changed from 15 to 60?
2. How did the learning rate affect the final cost and training behavior?
3. Why does a lower threshold usually predict more records as attacks?
4. Why does a lower final cost indicate that the model's probability estimates improved?
5. For your chosen `NEW_RECORD`, what attack probability did the model produce, and why did that probability result in its predicted label?

**Reminder:** Training accuracy is measured on the same synthetic records used to train the model. It is not a measure of performance on unseen real-world network traffic.